# 01.-Carga de Bases Marine - PROGRAM 01

##  Overview
In this program, the paths to the BDX files, which are the main input, are defined.
The information is loaded and consolidated into a standardized format.
The actuarial information is then loaded, and a classification of the updateable and legacy records is made.
The information is cross-referenced, resulting in the df_update database.
The df_update database contains all the records that will be updated with the BDX information for the update year/month.

##  Execution Flow
1. Library Import.
2. Path Definition and Macrovariables. 
3. Data import(Original BDX). 
4. Functions Definition.
5. Transformation of the original BDX.
6. Creation of BDX database.
7. Creation of monthly records database.
8. Import and format over previous monthly processed database.
9. First Validation of Gross Reserve and Deductible.
10. Merge Update Database and BDX Database.
11. Append of update DB and Monthly DB.
12. Second Validation: Dates and deductible.
13. Final appends and format.
14. Final Export.

##  Output
- Excel file with the update database of the month. (bd_update.xlsx)
- Pickle file with the update database of the month. (bd_update.pkl)

## 1. Library Import

In [39]:
# =============================================================================
# Section 1: Library import
# =============================================================================

# === Optional: Clean all the enviroment prior running === #
%reset -f                                                  
# ======================================================== #
import os
import pandas as pd
#import dtale
import numpy as np
import timeit
import sqlite3
import calendar

## 2. Path Definition and Macrovariables

In [40]:
# =============================================================================
# Section 2: Path Definition and Macrovariables. 
# =============================================================================
start_time = timeit.default_timer() # Timer
#================= Edit variables month================================


AñoMes = 202510 # This variable represent the month pf processing
AñoMes_ant = 202509  # This variable represents the previous period for wich we have information

# ========== Generated variables ================= #
#Year and month
año = AñoMes // 100  # Year, 4 digits
mes = AñoMes % 100   # Month, 2 digits
# Generate the names of the months
str1_month = calendar.month_name[mes].upper()  # Name of the month, all capital letters
str2_month = calendar.month_name[mes].capitalize()  # First letter in capital
# Replace the numbers from emglish to spanish
nombre_meses = {
    'JANUARY': 'ENERO', 'FEBRUARY': 'FEBRERO', 'MARCH': 'MARZO', 'APRIL': 'ABRIL',
    'MAY': 'MAYO', 'JUNE': 'JUNIO', 'JULY': 'JULIO', 'AUGUST': 'AGOSTO',
    'SEPTEMBER': 'SEPTIEMBRE', 'OCTOBER': 'OCTUBRE', 'NOVEMBER': 'NOVIEMBRE', 'DECEMBER': 'DICIEMBRE'
}

str1_month = nombre_meses.get(str1_month, str1_month)  # Spanish conversion, all capitals
str2_month = str1_month.capitalize()  # First letter as capital
# ================================================ #


# Convert the variables in datetime 
date_start_period = pd.to_datetime(str(AñoMes_ant), format='%Y%m') + pd.offsets.MonthBegin(1) # This is an Aux Variable and it should be used when we are proccesing more than 1 month
date_end_period = pd.to_datetime(str(AñoMes), format='%Y%m') + pd.offsets.MonthEnd(1) # This is an Aux Variable and it should be used when we are proccesing more than 1 

print("==============================================================================================")
print(f'Fecha de inicio para considerar en la base mensual: {date_start_period}')
print(f'Fecha de fin para considerar en la base mensual: {date_end_period}')
print("==============================================================================================")
# ================= Path definition =========================================

# Define your project directory path as a variable
directorio_proyecto =  "C:/Users/KOT14/Documents/Integral/Marine"   #C:/Users/KOT23/Documents/Integral

# Change the current working directory to the project directory
os.chdir(directorio_proyecto) 

# Verify that the current directory is the project directory
print(f"Directorio actual de trabajo: {os.getcwd()}")

# Main folders inside project directory
carpetas_principales = ["Bases", "Incidencias", "Insumos", "Procesados", "Validaciones"]

# Folders that will require the creation of a new folder
carpetas_con_añomes = ["Incidencias", "Insumos", "Procesados", "Validaciones"]

# Creation of the main folders if dont't exist
for carpeta in carpetas_principales:
    ruta_carpeta = os.path.join(directorio_proyecto, carpeta)
    if not os.path.exists(ruta_carpeta):
        os.makedirs(ruta_carpeta)
        print(f"Directorio principal creado: {ruta_carpeta}")
    else:
        print(f"El directorio principal ya existe: {ruta_carpeta}")

# Add a subfolder in the carpetas_con_añomes folders
for carpeta in carpetas_con_añomes:
    ruta_completa = os.path.join(directorio_proyecto, carpeta, str(AñoMes))
    if not os.path.exists(ruta_completa):
        os.makedirs(ruta_completa)
        print(f"Subdirectorio AñoMes creado: {ruta_completa}")
    else:
        print(f"El subdirectorio AñoMes ya existe: {ruta_completa}")

# Processed files path
ruta_procesados = rf"{directorio_proyecto}/Procesados/{AñoMes}" 
print(f"Ruta de archivos procesados: {ruta_procesados}")

# Incidences files path
ruta_incidencias = rf"{directorio_proyecto}/Incidencias/{AñoMes}" 
print(f"Ruta de Incidencias: {ruta_incidencias}")

# Validation files path
route_validations = rf"{directorio_proyecto}/Validaciones/{AñoMes}" 
print(f"Ruta de Validaciones: {route_validations}")

# Catalog files path
route_catalog = rf'{directorio_proyecto}/Catalogos'
print(f'Ruta de catalogos: {route_catalog}')

# Actuarial database path
route_actuarial = rf'{directorio_proyecto}/Bases/{AñoMes_ant}' 
print(f'Ruta de base actuarial: {route_actuarial}')

# Cleaning of the columnas list
del carpetas_con_añomes,carpetas_principales

Fecha de inicio para considerar en la base mensual: 2025-10-01 00:00:00
Fecha de fin para considerar en la base mensual: 2025-10-31 00:00:00
Directorio actual de trabajo: C:\Users\KOT14\Documents\Integral\Marine
El directorio principal ya existe: C:/Users/KOT14/Documents/Integral/Marine\Bases
El directorio principal ya existe: C:/Users/KOT14/Documents/Integral/Marine\Incidencias
El directorio principal ya existe: C:/Users/KOT14/Documents/Integral/Marine\Insumos
El directorio principal ya existe: C:/Users/KOT14/Documents/Integral/Marine\Procesados
El directorio principal ya existe: C:/Users/KOT14/Documents/Integral/Marine\Validaciones
El subdirectorio AñoMes ya existe: C:/Users/KOT14/Documents/Integral/Marine\Incidencias\202510
El subdirectorio AñoMes ya existe: C:/Users/KOT14/Documents/Integral/Marine\Insumos\202510
El subdirectorio AñoMes ya existe: C:/Users/KOT14/Documents/Integral/Marine\Procesados\202510
El subdirectorio AñoMes ya existe: C:/Users/KOT14/Documents/Integral/Marine\Va

## 3. Data import

In [41]:
# =============================================================================
# Section 3: Data import
# =============================================================================
# Raw data path
#path_bdx_1 = f"{directorio_proyecto}/Insumos/Legacy/AXA_ING_Carga_{año}.xlsx" # Ya no se usa
#path_bdx_2 = f"{directorio_proyecto}/Insumos/Legacy/AXA_ING_Casco_y_M2_{año}.xlsx"  # Ya no se usa
#path_bdx_3 = f"{directorio_proyecto}/Insumos/{AñoMes}/Bordereaux_Inbursa_Casco_Pandi_{str1_month}_{año}.xls"  # Bordereaux_Inbursa_Casco_Pandi_OCTUBRE_2024 -- Bordereaux_Inbursa_Casco_Pandi_JULIO_2024
#path_bdx_4 = f"{directorio_proyecto}/Insumos/Legacy/GNP_BDX_13637426_{str1_month}_{año}.xlsx" # Ya no se usa  
#path_bdx_5 = f"{directorio_proyecto}/Insumos/Legacy/GNP_BDX_38417374_{str1_month}_{año}.xlsx" # Ya no se usa
path_bdx_6 = f"{directorio_proyecto}/Insumos/{AñoMes}/Bordereaux_Inbursa_Casco_Pandi_Transporte_09-11_{str1_month}_{año}.xlsx" # Bordereaux_Inbursa_Casco_Pandi_Transporte_09-11_OCTUBRE_2024 -  Bordereaux_Inbursa_Casco_Pandi_Transporte_09-11_JULIO_2024
path_bdx_7 = f"{directorio_proyecto}/Insumos/{AñoMes}/Bordereaux_Inbursa_Casco_Pandi_Transporte_11-13_{str1_month}_{año}.xlsx" # Bordereaux_Inbursa_Casco_Pandi_Transporte_11-13_OCTUBRE_2024 -- Bordereaux_Inbursa_Casco_Pandi_Transporte_11-13_JULIO_2024 
path_bdx_8 = f"{directorio_proyecto}/Insumos/{AñoMes}/Bordereaux_Inbursa_Casco_Pandi_Transportes_13-15_{str1_month}_{año}.xlsx" # Bordereaux_Inbursa_Casco_Pandi_Transportes_13-15_OCTUBRE_2024 --Bordereaux_Inbursa_Casco_Pandi_Transportes_13-15_JULIO_2024 
path_bdx_9 = f"{directorio_proyecto}/Insumos/Legacy/Bordereaux_Inbursa_Aguas Profundas_12-14_{str1_month}_{año}.xlsx" 
path_bdx_10 = f"{directorio_proyecto}/Insumos/{AñoMes}/BDX PEMEX NCGL-070-1000773 - {str1_month} {año}  Casco PandiTransporte y Plataformas.xlsx" # Banorte_BDX PEMEX NCGL-070-1000773 - OCTUBRE 2024  Casco PandiTransporte y Plataformas --Banorte_BDX PEMEX NCGL-070-1000773 - JULIO 2024  Casco PandiTransporte y Plataformas
path_bdx_11 = f"{directorio_proyecto}/Insumos/{AñoMes}/Bordereaux_Seguros Atlas E01-2-60-3_{str2_month} {año}.xlsx" # Bordereaux_Seguros Atlas E01-2-60-3_Octubre  2024 -- Bordereaux_Seguros Atlas E01-2-60-3_Julio 2024
path_bdx_12 = f"{directorio_proyecto}/Insumos/{AñoMes}/BDX Pemex NCGL-070-1002258 REAS {str1_month} {año}.xlsx" # Bordereaux_Inbursa_Plataformas_14-16_OCTUBRE_2024 -- Bordereaux_Inbursa_Plataformas_14-16_JULIO_2024
path_bdx_13 = f"{directorio_proyecto}/Insumos/{AñoMes}/Bordereaux_Seguros Atlas E01-2-60-10_ {str2_month} {año}.xlsx" #Bordereaux_Seguros Atlas E01-2-60-10_ Octubre  2024 -- Bordereaux_Seguros Atlas E01-2-60-10_ Julio 2024
path_bdx_14 = f"{directorio_proyecto}/Insumos/{AñoMes}/Bordereaux_Mapfre_Casco_Pandi_Transportes_21-23_{str1_month}_{año}.xlsx" # Bordereaux_Mapfre_Casco_Pandi_Transportes_21-23_oct_2024 -- Bordereaux_Mapfre_Casco_Pandi_Transportes_21-23_Jul_2024

## 4. Functions definition

In [42]:
# =============================================================================
# Section 4: Functions definition
# =============================================================================

# Function to read the files and cut after the word "FIN"
def import_format(df):
    # Verify if "#" is in some of the rows of the first column
    header_row = df[df.iloc[:, 0] == '#'].index[0]  # Encontrar la fila donde está "#"
    
    # Redefine the dataframe using the row of the header
    df.columns = df.iloc[header_row]
    df = df.drop(index=range(header_row + 1))  # Drop rows before header
    df = df.reset_index(drop=True)

    # Delete rows after 'FIN'
    if '#' in df.columns:
        
        df['#'] = df['#'].astype(str).str.strip()
        
        fin_mask = df['#'] == 'FIN'
        
        if fin_mask.any():  # Solo si existe FIN
            end_index = df[fin_mask].index[0]
            df = df.iloc[:end_index]
        else:
            print("⚠️ No se encontró 'FIN' en esta hoja")
    
    return df

    # Reset the indexes and return the cleaned dataframe
    return df.reset_index(drop=True)  # Reset the indexes

# Silence the copy warnings
pd.options.mode.chained_assignment = None

## 5. Policies 

In [43]:
# INBURSA POLICIES FROM 2009-2011
hojas_dict_6 = pd.read_excel(path_bdx_6, sheet_name=None, header = None) #This returns a dictionary of dataframes, where each key is a dataframe
columns_selected = ['#', 'INWARD POLICY N°','REF. INB','REF. PEMEX','SUBSIDIARY','D.O.L.','D.O.R.','GROSS RESERVE','DEDUCTIBLE', 'COVERAGE','LOCATION','DESCRIPTION','STATUS','INBURSA OBSERVATIONS','KOT OBSERVATIONS']

# Desagregate the dataframes
df_30002933casco_bdx = import_format(hojas_dict_6['Casco 09-11'])
df_30002933casco_bdx['INWARD POLICY N°'] = '25200 30002933' 
df_30002933casco_bdx = df_30002933casco_bdx[columns_selected]
# 
df_30002933pandi_bdx = import_format(hojas_dict_6['Pandi 09-11'])
df_30002933pandi_bdx['INWARD POLICY N°'] = '25200 30002933' 
df_30002933pandi_bdx = df_30002933pandi_bdx[columns_selected]
#
df_30002933trans_bdx = import_format(hojas_dict_6['Transporte 09-11'])
df_30002933trans_bdx['INWARD POLICY N°'] = '25200 30002933'
df_30002933trans_bdx = df_30002933trans_bdx[columns_selected] 
# Append the data from the processed df to the consolidated df
df_bdx_6 = pd.concat([df_30002933casco_bdx, df_30002933pandi_bdx, df_30002933trans_bdx], ignore_index=True)
df_bdx_6.rename(columns= {'REF. INB':'CLAIM NUMBER', 'D.O.L.':'DATE OF LOSS','D.O.R.' : 'DATE OF REPORT','INBURSA OBSERVATIONS':'CEDANT OBSERVATIONS' }, inplace= True )
# Cleaning
del hojas_dict_6,df_30002933casco_bdx, df_30002933pandi_bdx, df_30002933trans_bdx
# Optional: export for debugging
#df_bdx_6.to_excel(fr'{ruta_procesados}/df_bdx_6.xlsx',index=False)


# INBURSA POLICIES FROM 2011-2013
hojas_dict_7 = pd.read_excel(path_bdx_7, sheet_name=None, header = None) #This returns a dictionary of dataframes, where each key is a dataframe
#Desagregate the dataframes
df_30006350casco_bdx = import_format(hojas_dict_7['Casco 11-13'])
df_30006350casco_bdx['INWARD POLICY N°'] = '25200 30006350' 
df_30006350casco_bdx = df_30006350casco_bdx[columns_selected]
# 
df_30006350pandi_bdx = import_format(hojas_dict_7['Pandi 11-13'])
df_30006350pandi_bdx['INWARD POLICY N°'] = '25200 30006350' 
df_30006350pandi_bdx = df_30006350pandi_bdx[columns_selected]
#
df_30006350trans_bdx = import_format(hojas_dict_7['Transporte 11-13'])
df_30006350trans_bdx['INWARD POLICY N°'] = '25200 30006350' 
df_30006350trans_bdx = df_30006350trans_bdx[columns_selected]
# Append the data from the processed df to the consolidated df
df_bdx_7 = pd.concat([df_30006350casco_bdx, df_30006350pandi_bdx, df_30006350trans_bdx], ignore_index=True)
df_bdx_7.rename(columns= {'REF. INB':'CLAIM NUMBER','D.O.L.' : 'DATE OF LOSS' ,'D.O.R.': 'DATE OF REPORT','INBURSA OBSERVATIONS':'CEDANT OBSERVATIONS'}, inplace= True )
# Cleaning
del hojas_dict_7,df_30006350casco_bdx, df_30006350pandi_bdx, df_30006350trans_bdx
# Optional: export for debugging
#df_bdx_7.to_excel(fr'{ruta_procesados}/df_bdx_7.xlsx',index=False)


# INBURSA POLICIES FROM 2013-2015
hojas_dict_8 = pd.read_excel(path_bdx_8, sheet_name=None, header = None) #This returns a dictionary of dataframes, where each key is a dataframe
#Desagregate the dataframes
df_30011610casco_bdx = import_format(hojas_dict_8['Casco 13-15'])
df_30011610casco_bdx['INWARD POLICY N°'] = '25300 30011610' 
df_30011610casco_bdx = df_30011610casco_bdx[columns_selected]
# 
df_30011610pandi_bdx = import_format(hojas_dict_8['Pandi 13-15'])
df_30011610pandi_bdx['INWARD POLICY N°'] = '25300 30011610' 
df_30011610pandi_bdx = df_30011610pandi_bdx[columns_selected]
#
df_30011610trans_bdx = import_format(hojas_dict_8['Transportes 13-15'])
df_30011610trans_bdx['INWARD POLICY N°'] = '25300 30011610'
df_30011610trans_bdx = df_30011610trans_bdx[columns_selected] 
# Append the data from the processed df to the consolidated df
df_bdx_8 = pd.concat([df_30011610casco_bdx, df_30011610pandi_bdx, df_30011610trans_bdx], ignore_index=True)
df_bdx_8.rename(columns= {'REF. INB':'CLAIM NUMBER','D.O.L.' : 'DATE OF LOSS' ,'D.O.R.': 'DATE OF REPORT','INBURSA OBSERVATIONS':'CEDANT OBSERVATIONS'}, inplace= True )
# Cleaning
del hojas_dict_8,df_30011610casco_bdx, df_30011610pandi_bdx, df_30011610trans_bdx
# Optional: export for debugging
#df_bdx_8.to_excel(fr'{ruta_procesados}/df_bdx_8.xlsx',index=False)

In [44]:
# LEGACY POLICIES DEEPWATERS
hojas_dict_9 = pd.read_excel(path_bdx_9, sheet_name=None, header = None) #This returns a dictionary of dataframes, where each key is a dataframe
columns_selecteddw = ['#', 'INWARD POLICY N°','REF. INB','REF. PEMEX','SUBSIDIARY','D.O.L.','D.O.R.','GROSS RESERVE','DEDUCTIBLE', 'COVER','LOCATION','DESCRIPTION','STATUS','INBURSA OBSERVATIONS','KOT OBSERVATIONS']

#Desagregate the dataframes
df_30008857dw_bdx = import_format(hojas_dict_9['AGUAS PROFUNDAS 12-14'])
df_30008857dw_bdx['INWARD POLICY N°'] = '25200 30008857' 
df_30008857dw_bdx = df_30008857dw_bdx[columns_selecteddw]
# Append the data from the processed df to the consolidated df
df_bdx_9 = pd.concat([df_30008857dw_bdx], ignore_index=True)
df_bdx_9.rename(columns= {'REF. INB':'CLAIM NUMBER','D.O.L.' : 'DATE OF LOSS' ,'D.O.R.': 'DATE OF REPORT','INBURSA OBSERVATIONS':'CEDANT OBSERVATIONS'}, inplace= True )
# Cleaning
del hojas_dict_9,df_30008857dw_bdx
# Optional: export for debugging
#df_bdx_9.to_excel(fr'{ruta_procesados}/df_bdx_9.xlsx',index=False)

In [45]:
# BANORTE POLICIES FROM 2017-2019
hojas_dict_10 = pd.read_excel(path_bdx_10, sheet_name=None, header = None) 
columns_selected = columns_selected = ['#','INWARD POLICY N°' ,'REF. BANORTE','REF. PEMEX','SUBSIDIARY','D.O.L.','D.O.R.','GROSS RESERVE','DEDUCTIBLE', 'COVERAGE','LOCATION','DESCRIPTION','STATUS','BANORTE OBSERVATIONS','KOT OBSERVATIONS']
# Desagregate the dataframes
df_NCGLcasco_bdx = import_format(hojas_dict_10['Casco 2017'])
df_NCGLcasco_bdx['INWARD POLICY N°'] = 'NCGL-070-1000773' 
df_NCGLcasco_bdx = df_NCGLcasco_bdx[columns_selected]
# 
df_NCGLpandi_bdx = import_format(hojas_dict_10['Pandi 2017'])
df_NCGLpandi_bdx['INWARD POLICY N°'] = 'NCGL-070-1000773' 
df_NCGLpandi_bdx = df_NCGLpandi_bdx[columns_selected]
#
df_NCGLtrans_bdx = import_format(hojas_dict_10['Transportes'])
df_NCGLtrans_bdx['INWARD POLICY N°'] = 'NCGL-070-1000773' 
df_NCGLtrans_bdx = df_NCGLtrans_bdx[columns_selected]
#
df_NCGLplat_bdx = import_format(hojas_dict_10['Plataformas Móviles'])
df_NCGLplat_bdx['INWARD POLICY N°'] = 'NCGL-070-1000773' 
df_NCGLplat_bdx = df_NCGLplat_bdx[columns_selected]
# Append the data from the processed df to the consolidated df
df_bdx_10 = pd.concat([df_NCGLcasco_bdx, df_NCGLpandi_bdx, df_NCGLtrans_bdx,df_NCGLplat_bdx], ignore_index=True)
df_bdx_10.rename(columns= {'REF. BANORTE':'CLAIM NUMBER', 'D.O.L.':'DATE OF LOSS','D.O.R.':'DATE OF REPORT','BANORTE OBSERVATIONS':'CEDANT OBSERVATIONS'}, inplace= True )
# Cleaning
del hojas_dict_10,df_NCGLcasco_bdx, df_NCGLpandi_bdx, df_NCGLtrans_bdx,df_NCGLplat_bdx
# Debug
#df_bdx_10.to_excel(fr'{ruta_procesados}/df_bdx_10.xlsx',index=False)

In [46]:
# ATLAS POLICIES FROM 2019-2021
hojas_dict_11 = pd.read_excel(path_bdx_11, sheet_name=None, header = None) 
columns_selected = columns_selected = ['#','INWARD POLICY N°','CLAIM NUMBER','REF. PEMEX','SUBSIDIARY','D.O.L.','D.O.R.','GROSS RESERVE','DEDUCTIBLE', 'COVERAGE','LOCATION','DESCRIPTION','STATUS','CEDANT OBSERVATIONS','KOT OBSERVATIONS']
# Desagregate the dataframes
df_atlas03pandi_bdx = import_format(hojas_dict_11['P&I_Pandi'])
df_atlas03pandi_bdx['INWARD POLICY N°'] = 'E01-2-60-000000003_0000-0-1' 
df_atlas03pandi_bdx = df_atlas03pandi_bdx[columns_selected]
# 
df_atlas03ferreo_bdx = import_format(hojas_dict_11['Ferreo'])
df_atlas03ferreo_bdx['INWARD POLICY N°'] = 'E01-2-60-000000003_0000-0-1' 
df_atlas03ferreo_bdx.rename( columns = { 'GROSS RESERVE (usd)':'GROSS RESERVE'  }, inplace= True) # Rename of the column
df_atlas03ferreo_bdx = df_atlas03ferreo_bdx[columns_selected]
#
df_atlas03casco_bdx = import_format(hojas_dict_11['Casco y Maquinaria'])
df_atlas03casco_bdx['INWARD POLICY N°'] = 'E01-2-60-000000003_0000-0-1' 
df_atlas03casco_bdx = df_atlas03casco_bdx[columns_selected]
#
df_atlas03flet_bdx = import_format(hojas_dict_11['Fletadores PMI RCcasco'])
df_atlas03flet_bdx['INWARD POLICY N°'] = 'E01-2-60-000000003_0000-0-1'
df_atlas03flet_bdx = df_atlas03flet_bdx[columns_selected]
#
df_atlas03carga_bdx = import_format(hojas_dict_11['Carga D y V'])
df_atlas03carga_bdx['INWARD POLICY N°'] = 'E01-2-60-000000003_0000-0-1' 
df_atlas03carga_bdx = df_atlas03carga_bdx[columns_selected]
# Append the data from the processed df to the consolidated df
df_bdx_11 = pd.concat([df_atlas03pandi_bdx, df_atlas03ferreo_bdx, df_atlas03casco_bdx,df_atlas03flet_bdx,df_atlas03carga_bdx], ignore_index=True)
df_bdx_11.rename(columns= {'D.O.L.':'DATE OF LOSS','D.O.R.' :'DATE OF REPORT'}, inplace= True)
# Cleaning
del hojas_dict_11,df_atlas03pandi_bdx, df_atlas03ferreo_bdx, df_atlas03casco_bdx,df_atlas03flet_bdx,df_atlas03carga_bdx
# Debug
#df_bdx_11.to_excel(fr'{ruta_procesados}/df_bdx_11.xlsx',index=False)

In [49]:
# MAPFRE POLICIES FROM 2021-2023
hojas_dict_12 = pd.read_excel(path_bdx_12, sheet_name=None, header = None) #This returns a dictionary of dataframes, where each key is a dataframe
columns_selected = columns_selected = ['#','INWARD POLICY N°','CLAIM NUMBER','REF. PEMEX','SUBSIDIARY','D.O.L.','D.O.R.','GROSS RESERVE','DEDUCTIBLE', 'COVERAGE','LOCATION','DESCRIPTION','STATUS','CEDANT OBSERVATIONS','KOT OBSERVATIONS']
#Desagregate the dataframes
df_mapfrecarga_bdx = import_format(hojas_dict_12['Carga B y P'])
df_mapfrecarga_bdx['INWARD POLICY N°'] = 'NCGL-070-1002258' 
df_mapfrecarga_bdx = df_mapfrecarga_bdx[columns_selected]
# 
df_mapfreferreo_bdx = import_format(hojas_dict_12['Ferreo'])
df_mapfreferreo_bdx['INWARD POLICY N°'] = 'NCGL-070-1002258' 
df_mapfreferreo_bdx = df_mapfreferreo_bdx[columns_selected]
#
df_mapfrepi_bdx = import_format(hojas_dict_12['P&I'])
df_mapfrepi_bdx['INWARD POLICY N°'] = 'NCGL-070-1002258' 
df_mapfrepi_bdx = df_mapfrepi_bdx[columns_selected]
#
df_mapfrecasco_bdx = import_format(hojas_dict_12['Casco'])
df_mapfrecasco_bdx['INWARD POLICY N°'] = 'NCGL-070-1002258'
df_mapfrecasco_bdx = df_mapfrecasco_bdx[columns_selected]
#
df_mapfreflet_bdx = import_format(hojas_dict_12['Fletadores RC PMI'])
df_mapfreflet_bdx['INWARD POLICY N°'] = 'NCGL-070-1002258' 
df_mapfreflet_bdx = df_mapfreflet_bdx[columns_selected]
#
df_mapfreplat_bdx = import_format(hojas_dict_12['PlataformasMoviles'])
df_mapfreplat_bdx['INWARD POLICY N°'] = 'NCGL-070-1002258' 
df_mapfreplat_bdx = df_mapfreplat_bdx[columns_selected]
#
df_mapfrecash_bdx = import_format(hojas_dict_12['Remesas y existencias de dinero'])
df_mapfrecash_bdx['INWARD POLICY N°'] = 'NCGL-070-1002258' 
df_mapfrecash_bdx = df_mapfrecash_bdx[columns_selected]
#
df_mapfreothers_bdx = import_format(hojas_dict_12['Antigüedades, obras de arte'])
df_mapfreothers_bdx['INWARD POLICY N°'] = 'NCGL-070-1002258'
df_mapfreothers_bdx = df_mapfreothers_bdx[columns_selected] 
# Append the data from the processed df to the consolidated df
df_bdx_12 = pd.concat([df_mapfrecarga_bdx, df_mapfreferreo_bdx, df_mapfrepi_bdx,df_mapfrecasco_bdx,df_mapfreflet_bdx,df_mapfreplat_bdx,df_mapfrecash_bdx,df_mapfreothers_bdx], ignore_index=True)
df_bdx_12.rename(columns= {'D.O.L.':'DATE OF LOSS', 'D.O.R.':'DATE OF REPORT'}, inplace= True)
# Cleaning
del hojas_dict_12,df_mapfrecarga_bdx, df_mapfreferreo_bdx, df_mapfrepi_bdx,df_mapfrecasco_bdx,df_mapfreflet_bdx,df_mapfreplat_bdx,df_mapfrecash_bdx,df_mapfreothers_bdx
# Optional: export for debugging
#df_bdx_12.to_excel(fr'{ruta_procesados}/df_bdx_12.xlsx',index=False)

In [50]:
# ATLAS POLICIES FROM 2023-2025
hojas_dict_13 = pd.read_excel(path_bdx_13, sheet_name=None, header = None) #This returns a dictionary of dataframes, where each key is a dataframe
columns_selected = columns_selected = ['#','INWARD POLICY N°','CLAIM NUMBER','REF. PEMEX','SUBSIDIARY','D.O.L.','D.O.R.','GROSS RESERVE','DEDUCTIBLE', 'COVERAGE','LOCATION','DESCRIPTION','STATUS','CEDANT OBSERVATIONS','KOT OBSERVATIONS']
# Desagregate the dataframes
df_atlas10casco_bdx = import_format(hojas_dict_13['Casco y Maquinaria'])
df_atlas10casco_bdx['INWARD POLICY N°'] = 'E01-2-60-000000010_0000-0-1' 
df_atlas10casco_bdx = df_atlas10casco_bdx[columns_selected]
# 
df_atlas10carga_bdx = import_format(hojas_dict_13['D Y V'])
df_atlas10carga_bdx['INWARD POLICY N°'] = 'E01-2-60-000000010_0000-0-1' 
df_atlas10carga_bdx = df_atlas10carga_bdx[columns_selected]
#
df_atlas10ferreo_bdx = import_format(hojas_dict_13['FERREO'])
df_atlas10ferreo_bdx['INWARD POLICY N°'] = 'E01-2-60-000000010_0000-0-1'  
df_atlas10ferreo_bdx = df_atlas10ferreo_bdx[columns_selected]
#
df_atlas10pmi_bdx = import_format(hojas_dict_13['PMI'])
df_atlas10pmi_bdx['INWARD POLICY N°'] = 'E01-2-60-000000010_0000-0-1' 
df_atlas10pmi_bdx = df_atlas10pmi_bdx[columns_selected]
#
df_atlas10pandi_bdx = import_format(hojas_dict_13['P&I_Pandi'])
df_atlas10pandi_bdx['INWARD POLICY N°'] = 'E01-2-60-000000010_0000-0-1' 
df_atlas10pandi_bdx = df_atlas10pandi_bdx[columns_selected]
# Append the data from the processed df to the consolidated df
df_bdx_13 = pd.concat([df_atlas10casco_bdx, df_atlas10carga_bdx, df_atlas10ferreo_bdx,df_atlas10pmi_bdx,df_atlas10pandi_bdx], ignore_index=True)
df_bdx_13.rename(columns= {'D.O.L.':'DATE OF LOSS', 'D.O.R.': 'DATE OF REPORT'} ,inplace= True)
# Cleaning
del hojas_dict_13,df_atlas10casco_bdx,df_atlas10carga_bdx,df_atlas10ferreo_bdx,df_atlas10pmi_bdx,df_atlas10pandi_bdx
# Debug
#df_bdx_13.to_excel(fr'{ruta_procesados}/df_bdx_13.xlsx',index=False)

In [51]:
# Inbursa Policy (Added at the end)
hojas_dict_14 = pd.read_excel(path_bdx_14, sheet_name=None, header = None) #This returns a dictionary of dataframes, where each key is a dataframe
columns_selected = columns_selected = ['#','INWARD POLICY N°','CLAIM NUMBER','REF. PEMEX','SUBSIDIARY','D.O.L.','D.O.R.','GROSS RESERVE','DEDUCTIBLE', 'COVERAGE','LOCATION','DESCRIPTION','STATUS','CEDANT OBSERVATIONS','KOT OBSERVATIONS']

#Desagregate the dataframes
df_mapfre2carga_bdx = import_format(hojas_dict_14['Carga B y P'])
df_mapfre2carga_bdx['INWARD POLICY N°'] = "'3612100000008"
df_mapfre2carga_bdx = df_mapfre2carga_bdx[columns_selected]
# 
df_mapfre2ferreo_bdx = import_format(hojas_dict_14['Ferreo'])
df_mapfre2ferreo_bdx['INWARD POLICY N°'] = "'3612100000008" 
df_mapfre2ferreo_bdx ="'" + df_mapfre2ferreo_bdx.astype(str)
df_mapfre2ferreo_bdx = df_mapfre2ferreo_bdx[columns_selected]
#
df_mapfre2pi_bdx = import_format(hojas_dict_14['P&I'])
df_mapfre2pi_bdx['INWARD POLICY N°'] = "'3612100000008"
df_mapfre2pi_bdx ="'" + df_mapfre2pi_bdx.astype(str)
df_mapfre2pi_bdx = df_mapfre2pi_bdx[columns_selected]
#
df_mapfre2casco_bdx = import_format(hojas_dict_14['Casco'])
df_mapfre2casco_bdx['INWARD POLICY N°'] = "'3612100000008"
df_mapfre2casco_bdx ="'" + df_mapfre2casco_bdx.astype(str)
df_mapfre2casco_bdx = df_mapfre2casco_bdx[columns_selected]
#
df_mapfre2flet_bdx = import_format(hojas_dict_14['FletadoresRC PMI'])
df_mapfre2flet_bdx['INWARD POLICY N°'] = "'3612100000008" 
df_mapfre2flet_bdx ="'" + df_mapfre2flet_bdx.astype(str)
df_mapfre2flet_bdx = df_mapfre2flet_bdx[columns_selected]
# Append the data from the processed df to the consolidated df
df_bdx_14 = pd.concat([df_mapfre2carga_bdx, df_mapfre2ferreo_bdx, df_mapfre2pi_bdx,df_mapfre2casco_bdx,df_mapfre2flet_bdx], ignore_index=True)
df_bdx_14.rename(columns= {'D.O.L.':'DATE OF LOSS', 'D.O.R.':'DATE OF REPORT'}, inplace= True)
# Cleaning
del hojas_dict_14,df_mapfre2carga_bdx, df_mapfre2ferreo_bdx, df_mapfre2pi_bdx,df_mapfre2casco_bdx,df_mapfre2flet_bdx
# Optional: export for debugging
#df_bdx_14.to_excel(fr'{ruta_procesados}/df_bdx_14.xlsx',index=False)

### 5.1 Debugging for NCGL-070-1002258

In [52]:
# =============================================================================
# DEBUGGING: Monitor de registros para la póliza NCGL-070-1002258
# =============================================================================

# Función para rastrear registros de la póliza específica
def check_policy_records(dataframe, policy_name, stage_name):
    """Función para contar y mostrar registros de una póliza específica en cada etapa"""
    count = len(dataframe[dataframe['INWARD POLICY N°'] == policy_name])
    print(f"[{stage_name}] Póliza {policy_name}: {count} registros")
    return count

# Inicializamos un diccionario para rastrear
tracking_policy = {}

# Rastrear después de consolidación de BDX
if 'df_bdx_consolidated' in locals():
    tracking_policy['After BDX Consolidation'] = check_policy_records(df_bdx_consolidated, 'NCGL-070-1002258', 'BDX Consolidation')


## 6. Creation of BDX Database


In [53]:
# =============================================================================
# Section 6: Creation of BDX database
# =============================================================================

# Consolidated Dataframe: concat only existing df_bdx_* variables
df_list = [globals()[f'df_bdx_{i}'] for i in range(6, 14) if f'df_bdx_{i}' in globals()]
if not df_list:
    raise RuntimeError('No se encontraron dataframes df_bdx_6..df_bdx_14 en el entorno. Revisa las lecturas anteriores.')
df_bdx_consolidated = pd.concat(df_list, ignore_index=True)

# Cleaning of individual dataframes: delete only existing keys
for i in range(1, 14):
    key = f'df_bdx_{i}'
    if key in globals():
        del globals()[key]
# Clening of list
del df_list

# Format variables
list_string = ['#','CLAIM NUMBER', 'REF. PEMEX','SUBSIDIARY', 'COVERAGE','LOCATION', 'DESCRIPTION',	'STATUS']
for col in list_string:
    df_bdx_consolidated[col] = df_bdx_consolidated[col].astype('string').str.strip()

# Numerical format
list_numeric = ['GROSS RESERVE', 'DEDUCTIBLE']
for col in list_numeric:
    df_bdx_consolidated[col] = pd.to_numeric(df_bdx_consolidated[col], errors='coerce')
    # Replace 0 with NaN, this is to simplify the conditions
    df_bdx_consolidated.loc[df_bdx_consolidated[col] == 0, col] = np.nan

# Optional: export for debugging                       
#df_bdx_consolidated.to_excel(fr'{ruta_procesados}/consolidado_full.xlsx' , index=False)       


# ======================== Cleaning of consolidated dataframe ======================== #

# ================= Known errors ================= #
# Move 'GROSS RESERVE' and 'DEDUCTIBLE' one row up if necessary
mask = (df_bdx_consolidated['CLAIM NUMBER'] == '35100 3041999') & df_bdx_consolidated[['GROSS RESERVE', 'DEDUCTIBLE']].isna().all(axis=1)
# Apply only if true
df_bdx_consolidated.loc[mask, ['GROSS RESERVE', 'DEDUCTIBLE']] = df_bdx_consolidated[['GROSS RESERVE', 'DEDUCTIBLE']].shift(-1) 
# Claim 51242741 #87, duplicated, we delete that record
df_bdx_consolidated = df_bdx_consolidated[~((df_bdx_consolidated['CLAIM NUMBER'] == '51242741') & (df_bdx_consolidated['#'] == '87'))].reset_index(drop= True) 


# ===================================================================================== #

# Quick Revision of records with empty CLAIM NUMBER
mask_review = (df_bdx_consolidated['CLAIM NUMBER'].isna()) & \
              ((~df_bdx_consolidated['GROSS RESERVE'].isna()) | (~df_bdx_consolidated['DEDUCTIBLE'].isna()))
# Creation of review dataframe
df_bdx_review_empty = df_bdx_consolidated[mask_review]
# Optional: export for debugging              
#df_bdx_review_empty.to_excel(fr'{ruta_procesados}/review_empty.xlsx', index= False)


# Visualize the records with error
print('Registros con CLAIM NUMBER vacío, pero con montos: ' )
print(df_bdx_review_empty[['INWARD POLICY N°','CLAIM NUMBER','DEDUCTIBLE']]) 
# Notes: Claim #3612100000008, this claim has a duplicated deductible 

# After the review, drop the records with empty claim number
# Drop records with empty claim number
df_bdx_consolidated = df_bdx_consolidated.loc[~df_bdx_consolidated['CLAIM NUMBER'].isna()].reset_index(drop=True)


# Optional: export for debugging              
#df_bdx_consolidated.to_excel(fr'{ruta_procesados}/consolidado.xlsx', index= False)

# ======================== Creation of Lob-Inward ======================== #
# Dictionary that assigns LoB inward according its value in Coverage 
mapping_dict = {
    'CASCO Y MAQ.': 'CASCO Y MAQUINARIA',   
    'CASCO': 'CASCO Y MAQUINARIA', 
    'Casco' : 'CASCO Y MAQUINARIA',
    'Casco y Maquinaria' : 'CASCO Y MAQUINARIA',
    'CASCO Y MAQUINARIA': 'CASCO Y MAQUINARIA',
    'Gastos de salvamento': 'CASCO Y MAQUINARIA',
    'Daños a la Maquinaria': 'CASCO Y MAQUINARIA',
    'Daños a la maquinaria' : 'CASCO Y MAQUINARIA',
    'Pandi': 'PANDI',
    'P&I': 'PANDI',
    'PANDI': 'PANDI',
    'CARGA': 'CARGO',
    'Carga' : 'CARGO',
    'Transporte': 'CARGO',
    'TRANSPORTE': 'CARGO',
    'TRANSPORTE / CONTAMINACION': 'CARGO',
    'POLIETILENO' : 'CARGA POLIETILENO',
    'Equipo Ferroviario': 'EQUIPO FERROVIARIO(DAÑO FÍSICO)',
    'FLETADORES PMI': 'RC FLETADORES(PMI)',
    'FletadoresPMI': 'RC FLETADORES(PMI)',
    'RC FLETADORES': 'RC FLETADORES(PEMEX)',
    'RC Fletadores' : 'RC FLETADORES(PEMEX)',
    'Aguas profundas': 'DEEP WATERS',
    'Aguas Profundas': 'DEEP WATERS'

}

# Create the column 'LoB-Inward' mapping the values in 'COVERAGE BDX'
df_bdx_consolidated['LoB-Inward'] = df_bdx_consolidated['COVERAGE'].map(mapping_dict)

# ========= Handmade Modifications ========= #
df_bdx_consolidated.loc[df_bdx_consolidated['CLAIM NUMBER'].isin(['35100 3049477','108638/2018','35100 3070073']), 'LoB-Inward'] = 'JACK-UPS(DAÑO FISICO)'
df_bdx_consolidated.loc[(df_bdx_consolidated['LoB-Inward']== 'CARGO') & df_bdx_consolidated['SUBSIDIARY'].isin(['Etileno', 'Pemex Etileno', 'ETILENO']), 'LoB-Inward'] = 'CARGA POLIETILENO'


# Claim number 621/2021 has duplicated deductible, delete the deductible of one of the records
df_bdx_consolidated.loc[(df_bdx_consolidated["#"] == "105.1") & (df_bdx_consolidated["CLAIM NUMBER"] == "621 / 2021" ), 'DEDUCTIBLE'] = 0

# Claim number = '51242741' y #87, duplicated claim in BDX
df_bdx_consolidated = df_bdx_consolidated[ ~((df_bdx_consolidated['CLAIM NUMBER'] == '51242741') & (df_bdx_consolidated ['#'] == '87'))]

# Claim Number 10205839-A, it should be 10205839
df_bdx_consolidated.loc[df_bdx_consolidated['CLAIM NUMBER'] == '10205839-A' , 'CLAIM NUMBER'] = '10205839'
# ========================================== #

# ========= Add the columns of policy period ========= #
dict_startdate = {
    'M9000325': '30/06/1999',
    'M9000324': '30/06/1999',
    'CJ200025': '30/06/2002',
    'BJ200008': '30/06/2002',
    'CJ2000250200': '30/06/2004',
    '90600 00320575': '30/06/2004',
    '90600 323484': '03/05/2005',
    '90600 328256': '20/02/2007',
    '25200 30002933': '20/02/2009',
    '25200 30006350': '20/02/2011',
    '25200 30008857': '31/08/2012',
    '25300 30011610': '20/02/2013',
    '147736177': '20/02/2015',
    'NCGL-070-1000773': '20/02/2017',
    'E01-2-60-000000003_0000-0-1': '20/02/2019',
    '3612100000008': '20/02/2021',
    'E01-2-60-000000010_0000-0-1': '20/02/2023',
    'NCGL-070-1002258': '20/02/2025'
}

dict_enddate = {
    'M9000325':'30/06/2002',
    'M9000324': '30/06/2002',
    'CJ200025': '30/06/2003',
    'BJ200008': '30/06/2003',
    'CJ2000250200': '30/06/2005',
    '90600 00320575': '30/06/2005',
    '90600 323484': '20/02/2007',
    '90600 328256': '20/02/2009',
    '25200 30002933': '20/02/2011',
    '25200 30006350': '20/02/2013',
    '25200 30008857':'31/12/2014',
    '25300 30011610': '20/02/2015',
    '147736177': '20/02/2017',
    'NCGL-070-1000773': '20/02/2019',
    'E01-2-60-000000003_0000-0-1': '20/02/2021',
    '3612100000008': '20/02/2023',
    'E01-2-60-000000010_0000-0-1': '20/02/2025',
    'NCGL-070-1002258': '20/02/2027'
}

# ==================================================== #

# ======================== Dates ======================== #
# Start and End date creation
df_bdx_consolidated['POLICY PERIOD START DATE'] = df_bdx_consolidated['INWARD POLICY N°'].map(dict_startdate)  
df_bdx_consolidated['POLICY PERIOD END DATE'] = df_bdx_consolidated['INWARD POLICY N°'].map(dict_enddate)

# Date columns
list_date = ['DATE OF LOSS','DATE OF REPORT','POLICY PERIOD START DATE','POLICY PERIOD END DATE']
for cols in list_date:
    df_bdx_consolidated[cols] = pd.to_datetime(df_bdx_consolidated[cols], dayfirst = True, errors='coerce')


# ======================== Format and cleaning ======================== #

# Delete all the spaces in 'CLAIM NUMBER'
df_bdx_consolidated['CLAIM NUMBER ORIGINAL'] = df_bdx_consolidated['CLAIM NUMBER']
df_bdx_consolidated['CLAIM NUMBER'] =  df_bdx_consolidated['CLAIM NUMBER'].str.strip().str.replace(' ', '', regex=False)
#Delete all the 0's at the begginin' of the policie CJ2000250200
mask = ((df_bdx_consolidated['INWARD POLICY N°'] == 'CJ2000250200') | (df_bdx_consolidated['INWARD POLICY N°'] == 'M9000324'))
df_bdx_consolidated.loc[mask, 'CLAIM NUMBER'] = df_bdx_consolidated.loc[mask, 'CLAIM NUMBER'].apply(lambda x: str(x).lstrip('0') if not pd.isna(x) else x)

# Duplicated Drop
# The following claim numbers have claim number duplicated but refering to the same claim, we drop the duplicated records
list_dup_bdx = ['8703/2019','4098/2020','3415/2019','4213/2019','244/2021','321361640000032','5727/2023',
                '3539/2019','322361640000011','323361640000021','1237/2020'
]

# Drop duplicated records in list_dup_bdx
df_bdx_consolidated = df_bdx_consolidated[~(
    (df_bdx_consolidated['CLAIM NUMBER'].isin(list_dup_bdx)) &  # If 'CLAIM NUMBER' is on the list 'list_dup_bdx'
    (df_bdx_consolidated.duplicated(subset=['CLAIM NUMBER'], keep='first'))  # Keep only the first record
)]

# Drop cover column
df_bdx_consolidated.drop(columns=['COVER'],errors='ignore', inplace=True)
# Rename of the column coverage to cover
df_bdx_consolidated.rename(columns={'COVERAGE': 'COVER'}, inplace=True)


Registros con CLAIM NUMBER vacío, pero con montos: 
      INWARD POLICY N° CLAIM NUMBER  DEDUCTIBLE
730     25200 30006350         <NA>   150298.88
2547  NCGL-070-1002258         <NA>     5000.00
2548  NCGL-070-1002258         <NA>     5000.00


Handmade Adjustment to have the information in case of a Deductible Variation, in general is prefered to modify the processed database from the previous month, as a working file                                                                                                               

In [54]:
# ===== Handmade Adjustments after identifying the variations in deductible ===== #

#if AñoMes == 202412:
#    df_bdx_consolidated.loc[(df_bdx_consolidated['CLAIM NUMBER'] == '6560/2020') & (df_bdx_consolidated['#'] == '13') , 'DEDUCTIBLE'] = 50000

# ======================== New Variables ======================== #
# Creation of variables that will help us cross the information
df_bdx_consolidated['KEY DED'] = df_bdx_consolidated['CLAIM NUMBER'] +  "-" + df_bdx_consolidated['DEDUCTIBLE'].fillna(0).astype(int).astype('string')

# Cleaning
#del mask_review, mapping_dict,dict_startdate,dict_startdate

# ============================================================================================ #
# Optional: export for debugging                                                                 #
df_bdx_consolidated.to_excel(fr'{ruta_procesados}/consolidado_bdx.xlsx', index= False)         #
df_bdx_consolidated.to_pickle(fr'{ruta_procesados}/consolidado_bdx.pkl')                       #
# ============================================================================================ #
    

## 7. Creation of monthly records database

In [55]:
# =============================================================================
# Section 7: Creation of monthly records database
# =============================================================================

# Select only the records that are reported in the month of process
df_monthly = df_bdx_consolidated[df_bdx_consolidated['DATE OF REPORT']>=date_start_period] 
# Selection of records with a possible mistake
df_monthly_errors = df_monthly[df_monthly['DATE OF LOSS']> date_end_period] 
# Show the records
if df_monthly_errors.empty:
    print('No tenemos registros con incidencia de fechas sobre la base mensual')
else:
    print('Los siguientes registros presentan incidencia en fecha en la base mensual(REVISAR):')
    print(df_monthly_errors[['CLAIM NUMBER', 'DATE OF LOSS','INWARD POLICY N°']]) # Select only relevant columns   

# Flag to identify monthly records
df_monthly['Monthly'] = 1
# == Format adjustments == #
df_monthly['LoB-Inward BDX'] = df_monthly['LoB-Inward']

# Drop the columns that are not needed
df_monthly.drop(columns=['#','KEY DED'] , inplace= True)

print('La base de registros mensuales considera el inicio:')
print(f'{date_start_period}')
print('La base de registros mensuales considera el fin:')
print(f'{date_end_period}')

No tenemos registros con incidencia de fechas sobre la base mensual
La base de registros mensuales considera el inicio:
2025-10-01 00:00:00
La base de registros mensuales considera el fin:
2025-10-31 00:00:00


## 8. Import and format over previous processed database

In [58]:
# =============================================================================
# Section 8: Import and format over previous processed database.
# =============================================================================

# ============= Uncomment if there´s an update ============= #
df_act_db = pd.read_excel(fr'{route_actuarial}/{AñoMes_ant}_Siniestros_Marine.xlsx') 
df_act_db.to_pickle(fr'{route_actuarial}/{AñoMes_ant}_Siniestros_Marine.pkl')

# ========================================================== #
# Optional
#df_act_db = pd.read_pickle(fr'{route_actuarial}/Marine_{AñoMes_ant}.pkl')


In [59]:
# ===== Creation of pivot database ===== #
""" 
This database is the pivot i.e we take this database as the source of all the records
This is made because the info in this database has been validated in the previous month
"""

# Select only Marine records
df_act_db = df_act_db[(df_act_db['INWARD POLICY'] == '02-Marine')]

# Delete all spaces (leading, trailing, and in between) in CLAIM NUMBER
df_act_db['CLAIM NUMBER ORIGINAL'] = df_act_db['CLAIM NUMBER'] # Keep the original Claim Number
df_act_db['CLAIM NUMBER'] = df_act_db['CLAIM NUMBER'].apply(lambda x: str(x).strip().replace(' ', '') if not pd.isna(x) else x)

# Date conversion
df_act_db['DATE OF LOSS'] = pd.to_datetime(df_act_db['DATE OF LOSS'], errors='coerce')
df_act_db['DATE OF REPORT'] = pd.to_datetime(df_act_db['DATE OF REPORT'], errors='coerce')

# Delete the records of  policies of subsidiary 
# Init the list of subsidiary companies
list_subsidiary = [
'25300 30021823','25200 30016456','25200 30028005','25300 30027961','25300 30028201','25300 30028443','25300 30031974','25200 30027587',
'E01-2-71-000001094_0000-0-0','E01-2-71-1103'
]

# Init the list of legacy bdx(meaning we don´t have new deliveries of information)
list_legacy = ['BJ200001','BJ2000120000','BJ2000120100','M9000325','M9000324','CJ200025','BJ200008','CJ2000250100', 'CJ2000250200','147736177','38417374','13637426']

# Dropping the records if INWARD POLICY N° in list_subsidiary
df_act_db = df_act_db.loc[~df_act_db['INWARD POLICY N°'].isin(list_subsidiary)].reset_index(drop= True) 
# This drop in claim number is justified in incidences 202407
df_act_db = df_act_db.loc[df_act_db['CLAIM NUMBER'] != '8434/2019'].reset_index(drop= True)

# Print the number of original records
print(f'Tenemos cargados {len(df_act_db)} registros originales cargados de la base actuarial')

# Creation of variables that will help us cross the information
df_act_db['KEY DED'] = df_act_db['CLAIM NUMBER'] + "-" + df_act_db['DEDUCTIBLE'].round(0).astype(int).astype('string')

Tenemos cargados 4355 registros originales cargados de la base actuarial


In [60]:
# RASTREO: Validar póliza específica después de procesar df_act_db
tracking_policy['After df_act_db load'] = check_policy_records(df_act_db, 'NCGL-070-1002258', 'After df_act_db load')

[After df_act_db load] Póliza NCGL-070-1002258: 156 registros


## 9. First Validation of Gross Reserve and Deductible

In [61]:
# =============================================================================
# Section 9: First Validation of Gross Reserve and Deductible
# =============================================================================


# ============== Actuarial Database ============== #
# We take the amounts in actuarial database
df_actdb_val1 = df_act_db.groupby(['CLAIM NUMBER', 'INWARD POLICY N°','DATE OF LOSS','DATE OF REPORT','LoB-Inward']).agg({
    'GROSS RESERVE': 'sum',
    'DEDUCTIBLE': 'sum',
    'Cumulative CLAIMS PAID': 'sum', # Aditional columns for conditionals
    'OSLR Inward': 'sum'
}).reset_index()

# Drop the legacy policies for the review
df_actdb_val1 = df_actdb_val1[~df_actdb_val1['INWARD POLICY N°'].isin(list_legacy)]

# ============== BDX Database ============== # 
# We take the amounts in BDX database
df_bdxdb_val1 = df_bdx_consolidated.groupby('CLAIM NUMBER').agg({
    'GROSS RESERVE': 'sum',
    'DEDUCTIBLE': 'sum'
}).reset_index()

# ========================================== # 
# Merge to compare numbers between databases
df_val1 = pd.merge(df_actdb_val1, 
                   df_bdxdb_val1, 
                      on = 'CLAIM NUMBER', 
                      how= 'left',
                      suffixes=('', ' BDX' ))

# ====================== Cross Review ====================== #
# List of known errors i.e identified records that don't cross and are awaiting an update
list_nocross = ['1826/2019', #"Se agrega la palabra '- Cancelado' al final del numero de siniestro
                '4650/2023', # "Se agrega el carácter ´.´al final del numero de siniestro
                'AGG-2017-2018-MKOTI000117','AGG-2018-2019-MKOTI000118', #The following claims have a recognized issue in deductible
                '351003048179', #This deductible was in MXN and should be in USD
                '63942114' # This record had an intern adjustment, we keep info of Actuarial DB
                ]

# Records that are NaN in GROSS RESERVE BDX and are not identified in list_nocross
df_rev1 = df_val1[(df_val1['GROSS RESERVE BDX'].isna()) & (~df_val1['CLAIM NUMBER'].isin(list_nocross))] 
# Review of the records that don´t cross
print("==============================================================================================")
print("Lista de Registros que no están cruzando entre Base actuarial y BDX, revisar:")
if df_rev1.empty:
    print('No hay registros que no crucen y no esten identificados')
else:
    print(df_rev1)
    df_rev1.to_excel(f'{ruta_procesados}/no_cruzan.xlsx', index= False)
    print('Archivo de registros que no cruzan exportado correctamente a excel')

# Cleaning
#del df_rev1

# ==================== Deductible variations review ==================== # 
# In this part of the process, if df_rev1 is empty we can fill the NaN with zero
df_val1[['GROSS RESERVE BDX', 'DEDUCTIBLE BDX']] = df_val1[['GROSS RESERVE BDX', 'DEDUCTIBLE BDX']].fillna(0)
df_val2 = df_val1.copy(deep=True) # Copy of the dataframe for the gross reserve deductible 

# Analyze the records that meet the folllowing criteria 
# -> Gross Reserve changed from 0 to positive in between months of analysis 
df_val1 = df_val1[((df_val1['GROSS RESERVE'].astype(int) == 0) & df_val1['GROSS RESERVE BDX'] > 0) | # Gross reserve changued from 0 to positive
                  ((df_val1['GROSS RESERVE'] > 0) & df_val1['GROSS RESERVE BDX'] > 0)  # Gross reserve > 0 in actuarial database
]  

# Deductible validation
df_val1['DEDUCTIBLE VALIDATION'] = (df_val1['DEDUCTIBLE'] - df_val1['DEDUCTIBLE BDX']).round(0) #Round to int
# Take the records that have diferences in the validation and that are not identifies en list_nocross
df_rev11 = df_val1[(df_val1['DEDUCTIBLE VALIDATION'] != 0) & (~df_val1['CLAIM NUMBER'].isin(list_nocross)) ]

print("==============================================================================================")
print('Análisis de variación de deducible:')
if df_rev11.empty:
    print('No hay registros con variación de deducible') # Claim 351003048179 is awaiting response
else:
    print('Registros con variación de deducible')
    df_rev11.to_excel(f'{ruta_procesados}/Validacion_deducible.xlsx', index= False) # Export
    print('Archivo de variaciones deducible exportado correctamente')
    print(df_rev11[['CLAIM NUMBER','DEDUCTIBLE VALIDATION']])
print("==============================================================================================")

# ==================== Gross reserve variations review ==================== # 

# List of exceptions
list_exc = ['63942114', # Correct amounts, 2 different covers
            '5726/2023' ] # Legacy Info

# Analyze the records that meet the folllowing criteria 
# -> Records with variation in Gross Reserve,CL payments > 0 and open reserve(OSLR > 0)
df_val2 = df_val2[
    (df_val2['GROSS RESERVE'].round(0) != df_val2['GROSS RESERVE BDX'].round(0)) & # Records with variation in Gross reserve
    (df_val2['Cumulative CLAIMS PAID'] > 0 ) & # Only consider records with claim payments
    (df_val2['OSLR Inward'].round(0) == 0) & # And records with no open reserve
    (~df_val2['CLAIM NUMBER'].isin(list_exc)) # Exclude the list in exceptions
]

# Rearrange of columns
df_val2 = df_val2[['CLAIM NUMBER','LoB-Inward', 'DATE OF LOSS', 'DATE OF REPORT', 'INWARD POLICY N°','GROSS RESERVE','GROSS RESERVE BDX', 
                   'OSLR Inward', 'Cumulative CLAIMS PAID']]

# ======== Display of the process ======== #
print('Análisis de variación de Gross Reserve:')
if df_val2.empty:
    print('No hay registros con variación de Gross Reserve') # Claim 351003048179 is awaiting response
else:
    print('Registros con variación de Gross Reserve:')
    df_val2.to_excel(f'{ruta_procesados}/Validacion_GR.xlsx', index= False) # Export
    print(df_val2[['CLAIM NUMBER','LoB-Inward' ,'INWARD POLICY N°', 'GROSS RESERVE', 'GROSS RESERVE BDX']]) 
    print('Archivo de variaciones Gross Reserve exportado correctamente')
print("==============================================================================================")


# ===== Cleaning ===== #
del list_exc


Lista de Registros que no están cruzando entre Base actuarial y BDX, revisar:
     CLAIM NUMBER INWARD POLICY N° DATE OF LOSS DATE OF REPORT  \
6     10003100524     90600 320575   2005-06-01     2005-06-09   
7     10003100931     90600 320575   2005-06-30     2005-07-01   
8     10003100950     90600 320575   2005-04-05     2005-07-05   
9     10003101768     90600 320575   2005-05-31     2005-07-11   
10    10003102629     90600 323484   2005-09-16     2005-10-14   
...           ...              ...          ...            ...   
4203      9981887         38417374   2008-03-10     2008-05-29   
4204      9981994         38417374   2008-03-02     2008-05-29   
4205      9982117         38417374   2008-03-25     2008-05-29   
4206      9982240         38417374   2008-03-28     2008-05-29   
4207      9982331         38417374   2008-03-14     2008-05-29   

              LoB-Inward  GROSS RESERVE  DEDUCTIBLE  Cumulative CLAIMS PAID  \
6     CASCO Y MAQUINARIA           0.00    100000.

In this part of the process we merge the df_act_db and df_bdx_consolidated
1st - Merge by the key "Claim Number" + "Deductible"
After this we have some records that dont´t cross, so we cross by claim number if there are no duplicated claims numbers, if we have a duplicated claim review manually


## 10. Merge Update Database and BDX Database

In [62]:
# =============================================================================
# Section 10: Merge Update Database and BDX Database
# =============================================================================
# Selection of the columns of each database
# Columns to select from df_update_db
columns_updb = ['CLAIM NUMBER ORIGINAL','CLAIM NUMBER', 'DATE OF LOSS', 'DATE OF REPORT','INWARD POLICY N°', 
                'SUBSIDIARY','LoB-Inward','COVER', 'COVERAGE' ,'KEY DED','GROSS RESERVE', 'DEDUCTIBLE',
                'Cumulative SALVAGE' ,'Cumulative EXPENSES PAID','Cumulative VALUATION EXPENSES',
                'Cumulative CLAIMS PAID','POLICY PERIOD START DATE','POLICY PERIOD END DATE','OSLR Inward',
                'KOT SHARE','Nat Cat'
]
# Columns to select from BDX database
columns_bdx = ['CLAIM NUMBER ORIGINAL','CLAIM NUMBER', 'INWARD POLICY N°', 'LoB-Inward', 'KEY DED','DATE OF LOSS', 'DATE OF REPORT','SUBSIDIARY',
               'GROSS RESERVE', 'DEDUCTIBLE', 'COVER', 'REF. PEMEX','LOCATION', 'DESCRIPTION', 'STATUS','POLICY PERIOD START DATE','POLICY PERIOD END DATE']


# === First Merge, using 'KEY DED' === #

# Initialize update database, this database is a step in the middle and will help us work in a safer enviroment
df_update_db = df_act_db.copy()
print(f'La base de actualización mensual comienza con {len(df_update_db)} registros')

# Agregar registros NUEVOS que existen en BDX pero no en la base actuarial
#new_records_bdx = df_bdx_consolidated[
#    ~df_bdx_consolidated['KEY DED'].isin(df_update_db['KEY DED'].astype(str).unique())
#].copy()

#if not new_records_bdx.empty:
#    print(f'Agregando {len(new_records_bdx)} registros nuevos del BDX')
    
    # Asegurar que las columnas coincidan
#   for col in columns_updb:
#        if col not in new_records_bdx.columns:
#            new_records_bdx[col] = None
    
    # Concatenar los nuevos registros
#    df_update_db = pd.concat([df_update_db, new_records_bdx[columns_updb]], ignore_index=True)

# Initialize the dataframe of manual adjustments
df_adj_manual = pd.DataFrame() # To save deleted records 
df_no_update = pd.DataFrame() # List of records that are not considered for update

# In this part of the process we delete all the records that are ok to not be found in df_bdx_consolidated
# We drop the records to have a correct merge and don't update unnecesary data

# =============== No update section  =============== #
# Claim 63942114, this claim had an adjustment in deductible, but we keep original info in acturial DB
df_no_update = pd.concat([df_no_update,df_update_db[df_update_db['CLAIM NUMBER'] == '63942114']])
df_no_update.loc[df_no_update['CLAIM NUMBER'] == '63942114','NOTAS'] = 'LEGACY INFO' 
df_update_db = df_update_db[~(df_update_db['CLAIM NUMBER'] == '63942114')] 

# Claim 112224/2017, this claim is canceled and has KEY DED repetated 
df_no_update = pd.concat([df_no_update,df_update_db[df_update_db['CLAIM NUMBER'] == '112224/2017']])
df_no_update.loc[df_no_update['CLAIM NUMBER'] == '112224/2017' , 'NOTAS'] = 'LEGACY INFO' 
df_update_db = df_update_db[~(df_update_db['CLAIM NUMBER'] == '112224/2017')] 

# Claim 322361640000001, this claim has justification
df_no_update = pd.concat([df_no_update,df_update_db[df_update_db['CLAIM NUMBER'] == '3223616400000']])
df_no_update.loc[df_no_update['CLAIM NUMBER'] == '3223616400000' , 'NOTAS'] = 'LEGACY INFO' 
df_update_db = df_update_db[~(df_update_db['CLAIM NUMBER'] == '3223616400000')] 

# All the claims in legacy_policies
df_no_update = pd.concat([df_no_update, df_update_db[df_update_db['INWARD POLICY N°'].isin(list_legacy)]])
df_no_update.loc[df_no_update['INWARD POLICY N°'].isin(list_legacy) ,'NOTAS'] = 'LEGACY POLICY' 
df_update_db = df_update_db[~(df_update_db['INWARD POLICY N°'].isin(list_legacy))] 

# The following claims because we take the original info in BDX
df_no_update = pd.concat([df_no_update,df_update_db[df_update_db['CLAIM NUMBER'].isin(['A0134500', 'A0719200', 'A0927399'])]])
df_no_update.loc[df_no_update['CLAIM NUMBER'].isin(['A0134500', 'A0719200', 'A0927399']) , 'NOTAS'] = 'LEGACY INFO' # We add a note over the file
df_update_db = df_update_db[~(df_update_db['CLAIM NUMBER'].isin(['A0134500', 'A0719200', 'A0927399']))] #

# The following claim because it has two rows of Gross reserve
df_no_update = pd.concat([df_no_update,df_update_db[df_update_db['CLAIM NUMBER'].isin(['108265/2018'])]]) # Add it to no update
df_no_update.loc[df_no_update['CLAIM NUMBER'].isin(['108265/2018']) , 'NOTAS'] = 'LEGACY INFO'
df_update_db = df_update_db[~(df_update_db['CLAIM NUMBER'].isin(['108265/2018']))] # Exclude the record from update_db

# Optional: export for debugging
#df_no_update.to_excel(f'{ruta_procesados}/no_update.xlsx', index= False)

# =============== Handamade update section  =============== #
# Identified records with typos
df_adj_manual = pd.concat([df_adj_manual,df_update_db[df_update_db['CLAIM NUMBER'].isin(list_nocross)]])
df_adj_manual.loc[df_adj_manual['CLAIM NUMBER']== '1826/2019' ,'NOTAS'] = 'Se espera corrección en BDX(TYPO)'
df_adj_manual.loc[df_adj_manual['CLAIM NUMBER']== '4650/2023' ,'NOTAS'] = 'Se espera corrección en BDX(TYPO)'
df_adj_manual.loc[df_adj_manual['CLAIM NUMBER'].isin(['AGG-2017-2018-MKOTI000117','AGG-2018-2019-MKOTI000118']) ,'NOTAS'] = 'MONTOS AGREGADOS'
df_adj_manual.loc[df_adj_manual['CLAIM NUMBER']== '351003048179' ,'NOTAS'] = 'Se espera corrección en BDX(DEDUCIBLE)'
df_update_db = df_update_db[~(df_update_db['CLAIM NUMBER'].isin(list_nocross))] 

# Records that are duplicated and are updated manually
list_dup_up = ['322361640000001-0' ,  #The folowing record is the only one with open reserve
'5726/2023-0'
]

# Drop records in list_dup_up
df_adj_manual = pd.concat([df_adj_manual, df_update_db[df_update_db['KEY DED'].isin(list_dup_up)]])
df_adj_manual.loc[df_adj_manual['KEY DED'].isin(list_dup_up),'NOTAS'] = 'Revisón por KEY DED repetido'
df_update_db = df_update_db[~(df_update_db['KEY DED'].isin(list_dup_up))]

# Optional: export for debugging
#df_adj_manual.to_excel(f'{ruta_procesados}/ajustes_manuales.xlsx', index= False)

# =============== Print of important information  =============== #

# Print the number of records after the cleaning
print(f'La base de actualización mensual termina con {len(df_update_db)} registros, despues de la limpieza')
print(f'La base de actualización manual tiene {len(df_adj_manual)} registros')
print(f'La base de registros que no se actualizan tiene {len(df_no_update)} registros')
print(f'Tenemos asi que la suma de la base limpia, la de actualizacion manual y la no actualizable es: {len(df_update_db) + len(df_adj_manual) + len(df_no_update)}')
#print('Base de  actualización manual exportada correctamente')

# Duplicated validation
# Filter for duplicated CLAIM NUMBER that are not in list_dup_up
df_dup_up = df_update_db[df_update_db.duplicated(subset=['KEY DED'], keep=False)]
# Print the records that are duplicated but not in list_dup_up
print('===============================================================================')
if df_dup_up.empty:
    print('No tenemos KEY DED repetidos en la base actuarial para el primer cruce')    
else:
    print(f'Tenemos las siguientes llaves duplicadas en df_updated_db que no están en list_dup_up:')
    print(df_dup_up)

# Review of the cross for df_bdx_consolidated
# Filter for duplicated CLAIM NUMBER that are not in list_dup_up
df_dup_bdx = df_bdx_consolidated[df_bdx_consolidated.duplicated(subset=['KEY DED'], keep=False)]
# Validation of duplicated recods on BDX database but only if they are considered in update database
df_dup_bdx = df_dup_bdx[df_dup_bdx['KEY DED'].isin(df_update_db['KEY DED'])]

# Print the records that are duplicated but not in list_dup_up
print('===============================================================================')
if df_dup_bdx.empty:
    print('No tenemos KEY DED repetidos en df_dup_bdx: para el primer cruce')    
else:
    print(f'Tenemos las siguientes llaves duplicadas en df_dup_bdx:')
    print(df_dup_bdx[['INWARD POLICY N°','CLAIM NUMBER','COVER','KEY DED']])

# Count the numbers of records in df_act_db before the merge
count_bm1 = len(df_update_db) # Count before merge 1
#Debugg
#df_update_db.to_excel(f'{ruta_procesados}/df_update_antesmerge.xlsx', index= False)

df_update_db['KEY DED'] = df_update_db['KEY DED'].astype(str)
df_bdx_consolidated['KEY DED'] = df_bdx_consolidated['KEY DED'].astype(str)

# Merge entre update database y BDX database
df_update_db = pd.merge(
    df_update_db[columns_updb],
    df_bdx_consolidated[columns_bdx],
    on='KEY DED',
#barby how='left'
    how='outer',
    suffixes=('', ' BDX')  # Sufijos más descriptivos
)


# Drop 'KEY DED' columns
#df_update_db.drop(['KEY DED'], axis = 1, inplace=True)

# Count the numbers of records after the merge
count_am1 = len(df_update_db) # Count after merge 1

# Print and review if the merge was correct
print('===============================================================================')
print(f'Teniamos {count_bm1} antes de merge 1 y tenemos {count_am1} al final del merge')

# Debug 1st merge
#df_update_db.to_excel(f'{ruta_procesados}/df_update_despuesmerge.xlsx', index= False)



La base de actualización mensual comienza con 4355 registros
La base de actualización mensual termina con 4294 registros, despues de la limpieza
La base de actualización manual tiene 18 registros
La base de registros que no se actualizan tiene 43 registros
Tenemos asi que la suma de la base limpia, la de actualizacion manual y la no actualizable es: 4355
Tenemos las siguientes llaves duplicadas en df_updated_db que no están en list_dup_up:
      YEAR OF THE RESERVE  ACCIDENT YEAR                      CLAIMS-ID  \
258                  2025         2025.0              101847/2025-CARGO   
259                  2025         2025.0              101847/2025-CARGO   
267                  2025         2025.0              101919/2025-CARGO   
268                  2025         2025.0              101919/2025-CARGO   
276                  2025         2025.0              102048/2025-CARGO   
...                   ...            ...                            ...   
4235                 2025      

In [63]:

cols_to_update = ['CLAIM NUMBER ORIGINAL', 'DATE OF LOSS', 'DATE OF REPORT', 'INWARD POLICY N°', 'DEDUCTIBLE', 'GROSS RESERVE']

# Suponiendo que df es tu DataFrame
for col in df_update_db.columns:
    if col.endswith(' BDX'):
        orig_col = col.replace(' BDX', '')
        # Solo rellenar si la columna original existe
        if orig_col in df_update_db.columns:
            df_update_db[orig_col] = df_update_db[orig_col].fillna(df_update_db[col])     

# Llenar CLAIM NUMBER con CLAIM NUMBER ORIGINAL si está vacío
if 'CLAIM NUMBER ORIGINAL' in df_update_db.columns and 'CLAIM NUMBER' in df_update_db.columns:
    df_update_db['CLAIM NUMBER'] = df_update_db['CLAIM NUMBER'].fillna(df_update_db['CLAIM NUMBER ORIGINAL'])                   

# New columns added from BDX 
cols_new = ['REF. PEMEX', 'LOCATION','DESCRIPTION', 'STATUS']

for col in cols_new:
    df_update_db[col] = df_update_db[col].combine_first(df_update_db[col])

# Fill NaN with 0
df_update_db[['GROSS RESERVE BDX','DEDUCTIBLE BDX']] = df_update_db[['GROSS RESERVE BDX','DEDUCTIBLE BDX']].fillna(0)

# Optional: export for debugging
#df_missing_cross.to_excel(f'{ruta_procesados}/nocross.xlsx' , index= False  )
df_update_db.to_excel(f'{ruta_procesados}/merge2.xlsx' , index= False)


## 11. Append of update DB and Monthly DB

In [64]:
# =============================================================================
# Section 11: Append of update DB and Monthly DB
# =============================================================================

# Flag to identify monthly records
df_update_db['Monthly'] = 0

# Validation
extra_columns = set(df_update_db.columns).difference(df_monthly.columns)
print(f'Tenemos que las columnas que están en la base actualizable y no están en la base mensual son: {extra_columns}')
# Concat Update Database and Monthly Database
print("Antes concat:", len(df_update_db))
df_update_db = pd.concat([df_update_db, df_monthly],ignore_index= True)
print("Después concat:", len(df_update_db))


Tenemos que las columnas que están en la base actualizable y no están en la base mensual son: {'CLAIM NUMBER BDX', 'Nat Cat', 'Cumulative EXPENSES PAID', 'KOT SHARE', 'OSLR Inward', 'GROSS RESERVE BDX', 'CLAIM NUMBER ORIGINAL BDX', 'COVER BDX', 'INWARD POLICY N° BDX', 'Cumulative VALUATION EXPENSES', 'KEY DED', 'DATE OF REPORT BDX', 'DEDUCTIBLE BDX', 'COVERAGE', 'Cumulative CLAIMS PAID', 'Cumulative SALVAGE', 'POLICY PERIOD END DATE BDX', 'POLICY PERIOD START DATE BDX', 'SUBSIDIARY BDX', 'DATE OF LOSS BDX'}
Antes concat: 4474
Después concat: 4479


In [65]:
# === Correction if needed === #

# This adjustment was becasue this is legacy info
if AñoMes == 202407:
    # Date of Loss
    df_update_db.loc[
        (~(df_update_db['CLAIM NUMBER'].isin(['737502','A0719200']))) & (df_update_db['Monthly'] == 0) ,
        'DATE OF LOSS'
    ] = df_update_db['DATE OF LOSS BDX']
    # Date of report
    df_update_db.loc[
       (~(df_update_db['CLAIM NUMBER'].isin(['737502','A0719200']))) & (df_update_db['Monthly'] == 0),
        'DATE OF REPORT'
    ] = df_update_db['DATE OF REPORT BDX']

## 12. Second validation: Dates and deductible

In [66]:
# =============================================================================
# Section 12: Second Validation: Dates and deductible
# =============================================================================


# Make a copy of df_update_db with only the records from previous months
df_incidences = df_update_db[df_update_db['Monthly'] == 0][[
    'CLAIM NUMBER', 'INWARD POLICY N°', 'SUBSIDIARY', 'GROSS RESERVE', 
    'GROSS RESERVE BDX', 'DATE OF LOSS', 'DATE OF LOSS BDX', 
    'DATE OF REPORT', 'DATE OF REPORT BDX', 'LoB-Inward', 'LoB-Inward BDX',
    'DEDUCTIBLE', 'DEDUCTIBLE BDX'
]].copy()

# Initialize validation columns
# Empty list
columns_incidences = []
# Iteration over the different columns
for col in ['LoB', 'DOL', 'DOR', 'DEDUCTIBLE']:     
    df_incidences[col + ' VALIDATION'] = 0
    #Create the list of columns_incidences 
    columns_incidences.append(col + ' VALIDATION')

# == Rearrange of columns == #
# Initialize the order
column_order = ['CLAIM NUMBER','INWARD POLICY N°','SUBSIDIARY', 'GROSS RESERVE', 'GROSS RESERVE BDX' ,'LoB-Inward','LoB-Inward BDX','LoB VALIDATION' ,'DATE OF LOSS','DATE OF LOSS BDX', 'DOL VALIDATION' ,'DATE OF REPORT','DATE OF REPORT BDX',
                'DOR VALIDATION','DEDUCTIBLE' ,'DEDUCTIBLE BDX', 'DEDUCTIBLE VALIDATION']
# Apply the order
df_incidences = df_incidences[column_order]
 
# 'LoB VALIDATION'
#df_incidences.loc[df_incidences['LoB-Inward'] != df_incidences['LoB-Inward BDX'] ,'LoB VALIDATION'] = 1 # Mark it if it's not the same as the month before
# == Dates incidences == #
list_incidences_dates = ['737502','A0719200'] # List of records that have legacy info 
# 'DOL VALIDAION' 
df_incidences['DOL VALIDATION'] = (
    (df_incidences['DATE OF LOSS BDX'] - df_incidences['DATE OF LOSS'])
    .dt.days
    .fillna(0)
    .astype(int)
)
# 'DOR VALIDATION'
df_incidences['DOR VALIDATION'] = (
    (df_incidences['DATE OF REPORT BDX'] - df_incidences['DATE OF REPORT'])
    .dt.days
    .fillna(0)
    .astype(int)
)
# Correction for legacy info
df_incidences = df_incidences[~df_incidences['CLAIM NUMBER'].isin(['737502', 'A0719200'])]

# Filter the records with incidences
df_with_variations = df_incidences[
    (df_incidences['DOL VALIDATION'] != 0) | (df_incidences['DOR VALIDATION'] != 0)
]

# Print the selected records
if not df_with_variations.empty:
    print('Registros con variaciones en fechas: ')
    print(df_with_variations[['CLAIM NUMBER', 'INWARD POLICY N°', 'DATE OF LOSS', 'DATE OF LOSS BDX', 'DOL VALIDATION', 'DOR VALIDATION']])
else:
    print('No hay registros con variaciones en fechas')



# ===== DEDUCTIBLE VALIDATION ===== #

# The deductible validation apply only for the records with open reserve 
# The deductible validation apply also for OSLR = 0 records
mask = (df_incidences['GROSS RESERVE'] != 0) | (df_incidences['GROSS RESERVE BDX'] != 0) 

df_incidences['DEDUCTIBLE VALIDATION'] = (
    df_incidences['DEDUCTIBLE BDX'].fillna(0).astype(int) -
    df_incidences['DEDUCTIBLE'].fillna(0).astype(int)
)
# Cleaning of the rows with no incidences 
# Keep the rows where there's at least one column in 'column_incidences' that's different from 0
df_incidences = df_incidences.loc[~(df_incidences[columns_incidences] == 0).all(axis=1)]

# Print the records with deductible incidences
if df_incidences.empty:
    print('No hay registros con variaciones en deducible')
else:
    print("Registros con incidencias en el deducible:")
    print(df_incidences.loc[df_incidences['DEDUCTIBLE VALIDATION'] != 0, ['CLAIM NUMBER', 'INWARD POLICY N°', 'DEDUCTIBLE', 'DEDUCTIBLE BDX', 'DEDUCTIBLE VALIDATION']])
    # Optional: export for debugging
    #df_incidences.to_excel(f'{ruta_incidencias}/Incidencias_Marine_{AñoMes}.xlsx',index= False) 

# ===== Cleaning ===== # 

del df_with_variations,df_incidences

Registros con variaciones en fechas: 
     CLAIM NUMBER             INWARD POLICY N° DATE OF LOSS DATE OF LOSS BDX  \
3263    4226/2024  E01-2-60-000000010_0000-0-1   2023-05-18       2023-05-22   

      DOL VALIDATION  DOR VALIDATION  
3263               4               0  
Registros con incidencias en el deducible:
     CLAIM NUMBER INWARD POLICY N°  DEDUCTIBLE  DEDUCTIBLE BDX  \
6     10003100524     90600 320575    100000.0             0.0   
7     10003100931     90600 320575    125000.0             0.0   
8     10003100950     90600 320575    125000.0             0.0   
9     10003101768     90600 320575    125000.0             0.0   
10    10003102629     90600 323484      1000.0             0.0   
...           ...              ...         ...             ...   
4467      9981887         38417374     10000.0             0.0   
4468      9981994         38417374     10000.0             0.0   
4469      9982117         38417374     10000.0             0.0   
4470      9982240   

## 13. Final appends and format

In [67]:
# =============================================================================
# Section 13: Final appends and format
# =============================================================================


columns_drop_final = ['CLAIM NUMBER BDX','DATE OF LOSS BDX','DATE OF REPORT BDX','INWARD POLICY N° BDX','SUBSIDIARY BDX','COVER BDX']

# Fill NaN with 0
df_update_db[['DEDUCTIBLE','GROSS RESERVE','DEDUCTIBLE BDX','GROSS RESERVE BDX','Cumulative SALVAGE','Cumulative EXPENSES PAID','Cumulative VALUATION EXPENSES','Cumulative CLAIMS PAID', 'OSLR Inward']] = \
    df_update_db[['DEDUCTIBLE','GROSS RESERVE','DEDUCTIBLE BDX','GROSS RESERVE BDX','Cumulative SALVAGE','Cumulative EXPENSES PAID','Cumulative VALUATION EXPENSES','Cumulative CLAIMS PAID','OSLR Inward']].fillna(0) 


df_update_db['KOT SHARE'] = df_update_db['KOT SHARE'].fillna(1)

df_update_db.drop(columns=columns_drop_final, inplace= True) 

# ===== Apend of the no update records and handmade adjustments ===== #

# Consolidated of dataframes
df_adj_manual = pd.concat([df_no_update,df_adj_manual],ignore_index= True)
# Clean up
del df_no_update

# == Drop extra columns from df_adj_manual == #
# Initialize the list of columns to drop
list_drop_manual =[
    'ID', 'XoL OSLR Retrocesion amount','CLAIMS-ID_XoL', 'KEY DED', 
    'SL OSLR Retrocesion amount', 'IBNR CEDED',  'XoL Paid Retrocesion amount', 
     'SL Paid Retrocesion amount', 'Net Incurred before Stop Loss', 'STOP LOSS SECTION',
    'Net Paid before Stop Loss','XoL Paid\n(Y/N)', 'XoL OSLR\n(Y/N)','StandRe LoB', 
    'UW Year', 'YEAR OF THE RESERVE', 'CLAIMS-ID', 'StandRe detailed LoB','ACCIDENT YEAR', 
    'ATTRITIONAL/LARGE','INWARD POLICY','Total Claims Paid no Alae', 
    'Total Claims Paid Inward', 'Inward Incurred', 'Inward Incurred no Alae'
]

# Filter the variales that exist in the dataframe
columns_to_drop = [col for col in list_drop_manual if col in df_adj_manual.columns]

# Drop the columns that exists in the dataframe
df_adj_manual.drop(columns=columns_to_drop, inplace=True)

# Validation
extra_columns = set(df_adj_manual.columns).difference(df_update_db.columns)
print(f'Tenemos que las columnas que están en el consolidado de no actualizables y no están en la base actualizable son: {extra_columns}')

print(f'Tenemos que la base de ajustes manuales tiene {len(df_adj_manual)} registros')
df_update_db = pd.concat([df_update_db,df_adj_manual], ignore_index= True)
# Fill NaN for 'NOTAS'
df_update_db['NOTAS'] = df_update_db['NOTAS'].astype(str).fillna('')
df_update_db['NOTAS'] = df_update_db['NOTAS'].replace('nan', '')
print(f'Tenemos que la base final cuenta con {len(df_update_db)} registros')

Tenemos que las columnas que están en el consolidado de no actualizables y no están en la base actualizable son: {'DEDUCTIBLE 202504', 'Cumulative EXPENSES PAID 202509', 'OSLR Inward 202504', 'DIF OSLR', 'NOTAS', 'Cumulative VALUATION EXPENSES 202504', 'Cumulative CLAIMS PAID 202504', 'GROSS RESERVE 202504', 'Cumulative EXPENSES PAID 202504', 'KEY LOB', 'Cumulative VALUATION EXPENSES 202509', 'FLAG CONTA', 'Cumulative CLAIMS PAID 202509'}
Tenemos que la base de ajustes manuales tiene 61 registros
Tenemos que la base final cuenta con 4540 registros


## 14. Final export 

In [68]:
# =============================================================================
# Section 14: Final Export
# =============================================================================

print(f'La base final de actualización tiene {sum(df_update_db['Monthly']==0)} registros pasados,{sum(df_update_db['Monthly']==1)} registros nuevos y {len(df_adj_manual)} de ajustes manuales ')
print(f'La base final de actualización termina con un total de {len(df_update_db)} registros')
# pkl file
df_update_db.to_pickle(f'{ruta_procesados}/bd_update.pkl')
# xlsx file
import re
df_update_db = df_update_db.map(
    lambda x: re.sub(r'[\x00-\x1f\x7f-\x9f]', '', x) if isinstance(x, str) else x
)
df_update_db.to_excel(f'{ruta_procesados}/bd_update.xlsx', index=False)



La base final de actualización tiene 4487 registros pasados,5 registros nuevos y 61 de ajustes manuales 
La base final de actualización termina con un total de 4540 registros


In [69]:
end_time = timeit.default_timer()
print(f"El programa terminó de correr correctamente en: {round(end_time - start_time,2)} s")

El programa terminó de correr correctamente en: 220.85 s
